## Imports

In [1]:
import os
import sys
import requests
import json
import time
import numpy as np

from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path
sys.path.insert(0, str(Path().resolve().parent))
from config import data_config
from preprocessing.preprocess_prices import preprocess_prices

## Load Data

In [ ]:
# Load .env file
load_dotenv()

# Get database URL (Alpha Vantage)
api_key = os.environ.get("AV_API_KEY")
base_url = os.environ.get("AV_BASE_URL")

# Get data
Path('../data').mkdir(parents=True, exist_ok=True)
for ticker in data_config.TICKERS:
    url = f"{base_url}function={data_config.AV_FUNCTION}&symbol={ticker}&apikey={api_key}"
    r = requests.get(url)
    data = r.json() 

    # Save data in JSON file
    with open(f'../data/daily_data_{ticker}.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    # Wait
    time.sleep(3)

    print(f"Daily {ticker} data saved to ../data/daily_data_{ticker}.json")


In [2]:
closing_prices_dict = {}

for ticker in data_config.TICKERS:
    # Load JSON file
    with open(f'../data/daily_data_{ticker}.json', 'r') as f:
        data = json.load(f)
    
    # Extract time series
    time_series = data['Time Series (Daily)']
    
    # Get closing prices sorted by date (oldest to newest)
    dates = sorted(time_series.keys())
    closing_prices = np.array([float(time_series[date]['4. close']) for date in dates])
    
    # Store in dictionary
    closing_prices_dict[ticker] = closing_prices
    
    # Print summary
    print(f"{ticker}:")
    print(f"  {len(closing_prices)} closing prices")
    print(f"  Date range: {dates[0]} to {dates[-1]}")
    print(f"  Price range: ${closing_prices.min():.2f} to ${closing_prices.max():.2f}")
    print(f"  Array shape: {closing_prices.shape}")
    print()


NVDA:
  100 closing prices
  Date range: 2025-09-12 to 2026-02-04
  Price range: $170.29 to $207.04
  Array shape: (100,)

GOOG:
  100 closing prices
  Date range: 2025-09-12 to 2026-02-04
  Price range: $237.49 to $344.90
  Array shape: (100,)

AAPL:
  100 closing prices
  Date range: 2025-09-12 to 2026-02-04
  Price range: $234.07 to $286.19
  Array shape: (100,)



## Preprocess Data

In [3]:
training_data = {}

for ticker in data_config.TICKERS:
    closing_prices = closing_prices_dict[ticker]
    training_data[ticker] = preprocess_prices(closing_prices, start=0)

print(training_data)


{'NVDA': (tensor([[ 0.0000,  1.4145, -0.2886, -0.0864, -0.5640, -1.3082, -0.6228,  0.0935],
        [-0.2000,  0.8955, -0.0844, -1.7979,  1.8927, -0.0775, -1.0419,  1.2361],
        [-0.5289,  1.6851, -0.8477,  0.6541,  1.1052, -0.5864,  0.3803, -0.9460],
        [-0.2223, -0.4597,  1.2873, -0.0209, -0.7771, -0.6391, -0.0268,  0.8502],
        [ 0.4865, -1.8666, -1.1055, -0.6249,  0.0383, -0.1638, -0.1606,  0.1348],
        [-0.7491,  2.0899,  2.2299, -0.3694,  0.3337, -0.1001, -0.3640, -0.6410],
        [ 1.5225, -0.7174, -0.6960, -1.1511,  2.6450,  1.2183,  1.1183, -1.7317],
        [-0.6812,  1.4262,  0.9764,  0.2401, -0.9570, -0.6852,  0.1018,  0.5385]]), np.float64(-1.5871549585307676), np.float64(341.12538949732624)), 'GOOG': (tensor([[ 0.0000, -0.2685, -0.2310,  0.2834,  0.1584, -0.8465, -0.0721,  1.2693],
        [ 1.2008, -0.9547, -2.2372, -1.1261,  1.3957, -0.9738,  2.3746,  0.1463],
        [ 1.0114,  0.6086,  0.3314, -0.2355,  0.2358,  0.3575,  1.0300,  0.0317],
        [ 1